In [1]:
import pandas as pd

In [2]:
training_data = pd.read_csv('data/v0.1/flextrack-2025-training-data-v0.1.csv', parse_dates=['Timestamp_Local'])
test_data = pd.read_csv('data/v0.1/flextrack-2025-public-test-data-v0.1.csv', parse_dates=['Timestamp_Local'])
random_preds = pd.read_csv('data/v0.1/flextrack-2025-random-prediction-data-v0.1.csv')

In [3]:
test_data_new = pd.read_csv('data/v0.2/flextrack-2025-public-test-data-v0.2.csv', parse_dates=['Timestamp_Local'])

In [4]:
training_data = pd.concat([training_data, test_data_new], axis=0).reset_index(drop=True)

In [5]:
time_col = "Timestamp_Local"
site_col = "Site"
index_cols = [site_col, time_col]
features = ["Dry_Bulb_Temperature_C", "Global_Horizontal_Radiation_W/m2", "Building_Power_kW"]
target_col_classification = "Demand_Response_Flag"
target_cols = [target_col_classification]
# target_cols = ["Demand_Response_Flag", "Demand_Response_Capacity_kW"]

### Create Data Splits

In [6]:
X_train = training_data[index_cols + features].copy()
y_train = training_data[index_cols + target_cols].copy()

### Create lags and rolling features

In [7]:
time_offsets = {"30min": 2, "1h": 4, "2h": 8, "6h": 4 * 6, "12h": 4 * 12, "1d": 24 * 4, "7d": 24 * 4 * 7, "m": 24 * 4 * 30, "y": 24 * 4 * 365}

In [8]:
def get_lags_features(df, lags: list = ["1h","1d","7d"], diffs: list = [1, 2, 4, 8]):
    stats = ['mean', 'std', 'min', 'max', 'median']
    lags_features = []
    grouped_df = df.groupby('Site')
    
    shifts = []
    for offset in lags:
        shift = df.set_index('Timestamp_Local').shift(freq=offset).reset_index().set_index(['Site',"Timestamp_Local"])
        shift.rename(columns=lambda x: f"{x}_shift_{offset}", inplace=True)
        shifts.append(shift)  

    windows_stats = []
    for window in lags:
        window_stats = []
        for stat in stats:
            window_stat = grouped_df.apply(
                lambda x: x.set_index('Timestamp_Local').rolling(window, min_periods=time_offsets[window]).agg(stat),
                include_groups=False
            ).reset_index().set_index(['Site',"Timestamp_Local"])
            window_stat.rename(columns=lambda x: f"{x}_roll_{window}_{stat}", inplace=True)
            window_stats.append(window_stat)
        windows_stats.append(pd.concat(window_stats, axis=1))
        
    diffs_dfs = []
    for diff in diffs:
        diffs_df = grouped_df.apply(
            lambda x: x.set_index('Timestamp_Local').diff(diff),
            include_groups=False
        ).reset_index().set_index(['Site',"Timestamp_Local"])
        diffs_df.rename(columns=lambda x: f"{x}_diff_{diff}", inplace=True)
        diffs_dfs.append(diffs_df)

        pct_change_df = grouped_df.apply(
            lambda x: x.set_index('Timestamp_Local').pct_change(diff),
            include_groups=False
        ).reset_index().set_index(['Site',"Timestamp_Local"])
        pct_change_df.rename(columns=lambda x: f"{x}_pct_change_{diff}", inplace=True)
        diffs_dfs.append(pct_change_df)

        # lags_features.append(pd.concat([*shifts, *window_stats, *diffs_dfs], axis=1))
    lags_features = pd.concat([*shifts, *windows_stats, *diffs_dfs], axis=1)
    return lags_features

In [9]:
import numpy as np

def add_time_features(df, time_col='Timestamp_Local'):
    df['Minute'] = df[time_col].dt.minute
    df['Hour'] = df[time_col].dt.hour
    df['Day'] = df[time_col].dt.day
    df['Month'] = df[time_col].dt.month
    df['Year'] = df[time_col].dt.year
    df['Day_Of_Week'] = df[time_col].dt.dayofweek
    df['Is_Weekend'] = df['Day_Of_Week'].isin([5, 6]).astype(int)
    
    # Enhanced cyclical features
    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Day_Of_Week_sin'] = np.sin(2 * np.pi * df['Day_Of_Week'] / 7)
    df['Day_Of_Week_cos'] = np.cos(2 * np.pi * df['Day_Of_Week'] / 7)
    
    # Hour categories for non-linear patterns
    df['Hour_Category'] = pd.cut(df['Hour'], bins=[-1, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'])
    
    # Season definition
    df['Season'] = df['Month'].map({12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3})
    return df

In [10]:
def create_features(df):
    df['Temp_by_Radiation'] = df['Dry_Bulb_Temperature_C'] * df['Global_Horizontal_Radiation_W/m2']
    df['Radiation_div_Temp'] = df['Global_Horizontal_Radiation_W/m2'] / df['Dry_Bulb_Temperature_C'] 
    df['Power_per_Temp'] = df['Building_Power_kW'] / (df['Dry_Bulb_Temperature_C'])
    return df

In [11]:
selected_lags = ["30min","1h", "2h", "6h", "12h", "1d", "7d"]
lags_features = get_lags_features(X_train, selected_lags)

In [12]:
site_stats = X_train.groupby(site_col)[features].agg(['mean', 'std', "median"])
site_stats.columns = [f"{col}_{stat}" for col, stat in site_stats.columns.values]
X_train = X_train.merge(site_stats, left_on=site_col, right_index=True)

In [13]:
X_train = create_features(X_train)

In [14]:
X_train = X_train.set_index(index_cols).join(lags_features).reset_index()

In [15]:
X_train = add_time_features(X_train)

In [16]:
X_train = X_train.set_index(index_cols)

In [17]:
y_train = y_train.set_index(index_cols)
y_train = y_train.loc[X_train.index]

In [18]:
extra_time_festures = ['Hour', 'Month', 'Year', 'Day_Of_Week']

### Train CatBoost Classifier
Implement CatBoost training using existing X_train and y_train, evaluate, show feature importances, and save the model.

In [19]:
# Install catboost if not available (uncomment if running in a fresh environment)
# %pip install catboost

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import joblib
import os, json

TARGET = "Demand_Response_Flag"

# Align and drop rows with missing target
train_df = X_train.join(y_train[[TARGET]])
train_df = train_df.dropna(subset=[TARGET])

# Encode multiclass labels (-1,0,1) to (0,1,2) for CatBoost if needed
classes_sorted = sorted(train_df[TARGET].unique().tolist())
class_to_idx = {c:i for i,c in enumerate(classes_sorted)}
idx_to_class = {i:c for c,i in class_to_idx.items()}
train_df['__label__'] = train_df[TARGET].map(class_to_idx)

# Time-based split (last 20% per site)
site_splits = []
for site, grp in train_df.groupby(level=0):
    n = len(grp)
    cut = int(n * 0.8)
    site_splits.append((grp.iloc[:cut], grp.iloc[cut:]))
train_part = pd.concat([p[0] for p in site_splits])
valid_part = pd.concat([p[1] for p in site_splits])

feature_cols = [c for c in train_part.columns if c not in [TARGET, '__label__']]

X_tr = train_part[feature_cols]
y_tr = train_part['__label__']
X_va = valid_part[feature_cols]
y_va = valid_part['__label__']

cat_features = ["Is_Weekend", "Hour_Category", "Season"]

train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
valid_pool = Pool(X_va, y_va, cat_features=cat_features)

label_dist = train_df[TARGET].value_counts(normalize=True)
class_weights = []
max_freq = label_dist.max()
for c in classes_sorted:
    freq = label_dist[c]
    class_weights.append(max_freq / freq)

model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    iterations=1200,
    # depth=10,
    learning_rate=0.0005,
    l2_leaf_reg=4,
    verbose=100,
    class_weights=class_weights,
    random_seed=321,
    # task_type="GPU",
    # devices='0',
    early_stopping_rounds=400
    
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

va_preds_idx = model.predict(X_va).astype(int).ravel()
va_probs = model.predict_proba(X_va)
va_preds = [idx_to_class[i] for i in va_preds_idx]
true_labels = [idx_to_class[i] for i in y_va]

print('Classes:', classes_sorted)
print('Accuracy:', accuracy_score(true_labels, va_preds))
print('Macro F1:', f1_score(true_labels, va_preds, average='macro'))
print('Weighted F1:', f1_score(true_labels, va_preds, average='weighted'))
print(classification_report(true_labels, va_preds))
print('Confusion Matrix:\n', confusion_matrix(true_labels, va_preds, labels=classes_sorted))

# Top feature importances
importances = model.get_feature_importance(train_pool)
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances}) \
    .sort_values('importance', ascending=False).head(40)
print(feat_imp)

os.makedirs('models', exist_ok=True)
model_path = 'models/catboost_classifier_multiclass.cbm'
model.save_model(model_path)
print(f'Model saved to {model_path}')

# Persist with joblib
joblib.dump(model, 'models/catboost_classifier_multiclass.joblib')

# Save class mapping
with open('models/catboost_class_mapping.json', 'w') as f:
    json.dump({'class_to_idx': class_to_idx, 'idx_to_class': idx_to_class}, f, indent=2)
print('Saved class mapping.')


0:	learn: 0.7848222	test: 0.6670436	best: 0.6670436 (0)	total: 125ms	remaining: 2m 30s
100:	learn: 0.8256545	test: 0.7424069	best: 0.7439806 (97)	total: 6.03s	remaining: 1m 5s
200:	learn: 0.8203545	test: 0.7404178	best: 0.7439806 (97)	total: 12s	remaining: 59.5s
300:	learn: 0.8210383	test: 0.7402696	best: 0.7439806 (97)	total: 18.2s	remaining: 54.4s
400:	learn: 0.8262961	test: 0.7403658	best: 0.7439806 (97)	total: 24.2s	remaining: 48.2s
Stopped by overfitting detector  (400 iterations wait)

bestTest = 0.7439806001
bestIteration = 97

Shrink model to first 98 iterations.
Classes: [-1.0, 0.0, 1.0]
Accuracy: 0.5773758561643836
Macro F1: 0.3164564746961997
Weighted F1: 0.7025066703552069
              precision    recall  f1-score   support

        -1.0       0.09      0.93      0.16       754
         0.0       1.00      0.57      0.72     27043
         1.0       0.04      0.70      0.07       235

    accuracy                           0.58     28032
   macro avg       0.37      0.73 

In [20]:
from imblearn.metrics import geometric_mean_score

geometric = geometric_mean_score(
    true_labels,
    va_preds,
    average="macro",
)
f1 = f1_score(
    true_labels,
    va_preds,
    average="macro",
)
print(f"Geometric: {geometric}, F1: {f1}")

Geometric: 0.7918010731465888, F1: 0.3164564746961997


### Test Predictions

In [21]:
X_test = test_data[index_cols + features].copy()

In [22]:
lags_features = get_lags_features(X_test, selected_lags)

In [23]:
X_test = X_test.set_index(index_cols).join(lags_features).reset_index()

In [24]:
X_test = add_time_features(X_test)

In [25]:
from catboost import CatBoostClassifier, Pool
import json, os

# Load model and class mappings
model_path = 'models/catboost_classifier_multiclass.cbm'
class_map_path = 'models/catboost_class_mapping.json'
assert os.path.exists(model_path), 'Model file not found.'
assert os.path.exists(class_map_path), 'Class mapping file not found.'

model = CatBoostClassifier()
model.load_model(model_path)
with open(class_map_path, 'r') as f:
    maps = json.load(f)
idx_to_class = {int(k):v for k,v in maps['idx_to_class'].items()}

# Prepare test feature matrix
X_test_feat = X_test.copy()
X_test_feat = X_test_feat.set_index(index_cols)

# Ensure alignment of features with training model
target_like_cols = ['Demand_Response_Flag', '__label__']
model_features = model.feature_names_  # order expected by model
# Drop any unexpected columns
X_test_feat_aligned = X_test_feat.drop(columns=[c for c in X_test_feat.columns if c not in model_features], errors='ignore')
# Add any missing features as NaN
for f in model_features:
    if f not in X_test_feat_aligned.columns:
        X_test_feat_aligned[f] = float('nan')
# Reorder
X_test_feat_aligned = X_test_feat_aligned[model_features]

cat_features = ["Is_Weekend", "Hour_Category", "Season"]

test_pool = Pool(X_test_feat_aligned, cat_features=cat_features)

pred_idx = model.predict(test_pool).astype(int).ravel()
probs = model.predict_proba(test_pool)

pred_labels = [idx_to_class[i] for i in pred_idx]

prob_cols = {f'proba_class_{idx_to_class[i]}': probs[:, i] for i in range(probs.shape[1])}

pred_df = X_test_feat_aligned.copy()
pred_df['Pred_Class'] = pred_labels
for k,v in prob_cols.items():
    pred_df[k] = v

# Reset index for export
pred_out = pred_df.reset_index()[index_cols + ['Pred_Class'] + list(prob_cols.keys())]

os.makedirs('models', exist_ok=True)
pred_csv_path = 'models/test_predictions.csv'
pred_out.to_csv(pred_csv_path, index=False)
print(f'Saved test predictions to {pred_csv_path}')

pred_out.head()

Saved test predictions to models/test_predictions.csv


,Site,Timestamp_Local,Pred_Class,proba_class_-1.0,proba_class_0.0,proba_class_1.0
0,siteD,2020-01-01 00:00:00,0.0,0.319763,0.360994,0.319243
1,siteD,2020-01-01 00:15:00,0.0,0.319802,0.360929,0.319269
2,siteD,2020-01-01 00:30:00,0.0,0.319471,0.361510,0.319019
3,siteD,2020-01-01 00:45:00,0.0,0.319644,0.361073,0.319284
4,siteD,2020-01-01 01:00:00,0.0,0.318532,0.362880,0.318588


In [26]:
pred_out.rename(columns={"Pred_Class":"Demand_Response_Flag"}, inplace=True)

In [27]:
pred_out[["Site","Timestamp_Local","Demand_Response_Flag"]].to_csv('data/submission-v1.0.csv', index=False)

In [28]:
true_test_data = pd.read_csv('data/v0.2/flextrack-2025-public-test-data-v0.2.csv', parse_dates=['Timestamp_Local'])

In [29]:
true_labels = true_test_data.set_index(["Site","Timestamp_Local"]).loc[pred_out.set_index(["Site","Timestamp_Local"]).index]['Demand_Response_Flag']
preds = pred_out['Demand_Response_Flag']

In [30]:
geometric = geometric_mean_score(
    true_labels,
    preds,
    average="macro",
)
f1 = f1_score(
    true_labels,
    preds,
    average="macro",
)
print(f"Geometric: {geometric}, F1: {f1}")

Geometric: 0.8044095930544235, F1: 0.36720883599284965


#### Test set for new phase

In [31]:
test_data_new = pd.read_csv('data/v0.2/flextrack-2025-public-test-data-v0.2.csv', parse_dates=['Timestamp_Local'])

In [32]:
test_data_new_missing_flag = test_data_new[test_data_new['Demand_Response_Flag'].isna()].copy()

In [33]:
X_test_missing_flag = test_data_new_missing_flag[index_cols + features].copy()

In [34]:
lags_features = get_lags_features(X_test_missing_flag, ["1h","1d","7d"])
X_test_missing_flag = X_test_missing_flag.set_index(index_cols).join(lags_features).reset_index()
X_test_missing_flag = add_time_features(X_test_missing_flag)

In [35]:
from catboost import CatBoostClassifier, Pool
import json, os

# Load model and class mappings
model_path = 'models/catboost_classifier_multiclass.cbm'
class_map_path = 'models/catboost_class_mapping.json'
assert os.path.exists(model_path), 'Model file not found.'
assert os.path.exists(class_map_path), 'Class mapping file not found.'

model = CatBoostClassifier()
model.load_model(model_path)
with open(class_map_path, 'r') as f:
    maps = json.load(f)
idx_to_class = {int(k):v for k,v in maps['idx_to_class'].items()}

# Prepare test feature matrix
X_test_feat = X_test_missing_flag.copy()
X_test_feat = X_test_missing_flag.set_index(index_cols)

# Ensure alignment of features with training model
target_like_cols = ['Demand_Response_Flag', '__label__']
model_features = model.feature_names_  # order expected by model
# Drop any unexpected columns
X_test_feat_aligned = X_test_feat.drop(columns=[c for c in X_test_feat.columns if c not in model_features], errors='ignore')
# Add any missing features as NaN
for f in model_features:
    if f not in X_test_feat_aligned.columns:
        X_test_feat_aligned[f] = float('nan')
# Reorder
X_test_feat_aligned = X_test_feat_aligned[model_features]

cat_features = ["Is_Weekend", "Hour_Category", "Season"]
test_pool = Pool(X_test_feat_aligned, cat_features=cat_features)

pred_idx = model.predict(test_pool).astype(int).ravel()
probs = model.predict_proba(test_pool)

pred_labels = [idx_to_class[i] for i in pred_idx]

prob_cols = {f'proba_class_{idx_to_class[i]}': probs[:, i] for i in range(probs.shape[1])}

pred_df = X_test_feat_aligned.copy()
pred_df['Pred_Class'] = pred_labels
for k,v in prob_cols.items():
    pred_df[k] = v

# Reset index for export
pred_out = pred_df.reset_index()[index_cols + ['Pred_Class'] + list(prob_cols.keys())]
pred_out.head()

,Site,Timestamp_Local,Pred_Class,proba_class_-1.0,proba_class_0.0,proba_class_1.0
0,siteE,2020-01-01 00:00:00,0.0,0.319763,0.360994,0.319243
1,siteE,2020-01-01 00:15:00,0.0,0.319763,0.360994,0.319243
2,siteE,2020-01-01 00:30:00,0.0,0.319432,0.361575,0.318992
3,siteE,2020-01-01 00:45:00,0.0,0.319432,0.361575,0.318992
4,siteE,2020-01-01 01:00:00,0.0,0.318416,0.363106,0.318478


In [37]:
test_data_new.set_index(['Site', "Timestamp_Local"], inplace=True)

In [38]:
test_data_new.loc[test_data_new['Demand_Response_Flag'].isna(),"Demand_Response_Flag"] = pred_out.set_index(['Site', "Timestamp_Local"])["Pred_Class"]

In [39]:
test_data_new['Demand_Response_Flag'].value_counts(dropna=False)

Demand_Response_Flag
 0.0    104149
-1.0       710
 1.0       261
Name: count, dtype: int64

In [40]:
test_data_new.reset_index().to_csv('data/v0.2/flextrack-2025-public-test-data-v0.2-with-flags.csv', index=False)